In [3]:
import pandas as pd

DATA_CSV = "data/dish.csv"

# === Загружаем данные ===
df = pd.read_csv(DATA_CSV)

df_train = df[df["split"] == "train"].reset_index(drop=True)
df_test  = df[df["split"] == "test"].reset_index(drop=True)

# === TRAIN ингредиенты ===
train_ingredients = set()
for ingr_str in df_train["ingredients"]:
    train_ingredients.update(ingr_str.split(";"))

# === TEST ингредиенты ===
test_ingredients = set()
for ingr_str in df_test["ingredients"]:
    test_ingredients.update(ingr_str.split(";"))

print("="*60)
print("📊 ОБЩАЯ СТАТИСТИКА TEST")
print("="*60)

print(f"Всего блюд (test): {len(df_test)}")
print(f"Всего уникальных ингредиентов (test): {len(test_ingredients)}")

# --- Блюда с 1 ингредиентом в test ---
single_ingr_test = df_test[df_test["ingredients"].str.count(";") == 0]
print(f"Блюд с 1 ингредиентом (test): {len(single_ingr_test)}")


📊 ОБЩАЯ СТАТИСТИКА TEST
Всего блюд (test): 507
Всего уникальных ингредиентов (test): 159
Блюд с 1 ингредиентом (test): 161


In [4]:
# === Ингредиенты в test, которых нет в train ===
test_only_ingredients = test_ingredients - train_ingredients

print("\n" + "="*60)
print("🚨 ИНГРЕДИЕНТЫ, КОТОРЫХ НЕТ В TRAIN")
print("="*60)

print(f"Всего уникальных ингредиентов в test, отсутствующих в train: {len(test_only_ingredients)}\n")

for ingr in sorted(test_only_ingredients):
    print(f"❌ {ingr}")



🚨 ИНГРЕДИЕНТЫ, КОТОРЫХ НЕТ В TRAIN
Всего уникальных ингредиентов в test, отсутствующих в train: 2

❌ ingr_0000000019
❌ ingr_0000000141


In [5]:
common_ingredients = train_ingredients & test_ingredients

print("\n" + "="*60)
print("📈 ПОКРЫТИЕ TEST ИНГРЕДИЕНТОВ TRAIN'ом")
print("="*60)

print(f"Ингредиентов в train: {len(train_ingredients)}")
print(f"Ингредиентов в test: {len(test_ingredients)}")
print(f"Общих ингредиентов: {len(common_ingredients)}")

coverage = len(common_ingredients) / len(test_ingredients) * 100 if test_ingredients else 0
print(f"Покрытие test ингредиентов train'ом: {coverage:.2f}%")



📈 ПОКРЫТИЕ TEST ИНГРЕДИЕНТОВ TRAIN'ом
Ингредиентов в train: 198
Ингредиентов в test: 159
Общих ингредиентов: 157
Покрытие test ингредиентов train'ом: 98.74%


In [7]:
import pandas as pd

DATA_CSV = "data/dish.csv"

# === Загружаем данные ===
df = pd.read_csv(DATA_CSV)

df_train = df[df["split"] == "train"].reset_index(drop=True)

# === TRAIN ингредиенты ===
train_ingredients = set()
for ingr_str in df_train["ingredients"]:
    train_ingredients.update(ingr_str.split(";"))

print("="*60)
print("📊 ОБЩАЯ СТАТИСТИКА TRAIN")
print("="*60)

print(f"Всего блюд (train): {len(df_train)}")
print(f"Всего уникальных ингредиентов (train): {len(train_ingredients)}")

# --- Блюда с 1 ингредиентом ---
single_ingr_train = df_train[df_train["ingredients"].str.count(";") == 0]
print(f"Блюд с 1 ингредиентом (train): {len(single_ingr_train)}")


📊 ОБЩАЯ СТАТИСТИКА TRAIN
Всего блюд (train): 2755
Всего уникальных ингредиентов (train): 198
Блюд с 1 ингредиентом (train): 749


In [6]:
from collections import defaultdict

# === Группы ингредиентов ===
ingredient_groups = {
    "meat_fish": [
        "beef","chicken","pork","lamb","turkey","duck","veal","bison",
        "fish","salmon","tuna","cod","shrimp","crab","lobster","sausage",
        "bacon","ham","steak","meat","ribs","wings","drumsticks"
    ],
    "vegetables": [
        "lettuce","spinach","broccoli","carrot","tomato","cucumber","onion",
        "pepper","zucchini","eggplant","cabbage","beans","peas","corn",
        "greens","kale","arugula","chard","celery","mushroom"
    ],
    "fruits": [
        "apple","banana","orange","pear","berry","berries","strawberry",
        "grape","melon","kiwi","pineapple","mango","peach","plum","fig"
    ],
    "sauces_spices": [
        "sauce","ketchup","mustard","mayonnaise","dressing","vinegar",
        "oil","olive oil","salt","pepper","garlic","herbs","spice",
        "soy sauce","pesto","gravy"
    ],
    "legumes": [
        "beans","lentils","chickpeas","peas","soy","tofu"
    ],
    "grains": [
        "rice","pasta","bread","oats","wheat","barley","quinoa","noodles",
        "cornmeal","couscous","bulgur"
    ],
    "alcohol": [
        "beer","wine","vodka","rum","whiskey","champagne","ale","lager"
    ],
    "drinks": [
        "juice","milk","smoothie","tea","coffee","soda","latte","shake"
    ],
    "water": [
        "water"
    ]
}

# === Функция определения группы ===
def detect_group(ingredient):
    ingr = ingredient.lower()
    for group, keywords in ingredient_groups.items():
        for kw in keywords:
            if kw in ingr:
                return group
    return "other"

# === Группировка ===
grouped = defaultdict(lambda: {"known": [], "unknown": []})

for ingr in all_ingredients:
    group = detect_group(ingr)
    if ingr in known_ingr_cal:
        grouped[group]["known"].append(ingr)
    else:
        grouped[group]["unknown"].append(ingr)

# === Печать ===
print("\n" + "="*60)
print("🧠 ИНГРЕДИЕНТЫ ПО ГРУППАМ")
print("="*60)

for group, data in grouped.items():
    print(f"\n🔹 ГРУППА: {group.upper()}")
    print(f"  Известна калорийность ({len(data['known'])}):")
    for i in sorted(data["known"]):
        print(f"    ✔ {i}")

    print(f"  Неизвестна калорийность ({len(data['unknown'])}):")
    for i in sorted(data["unknown"]):
        print(f"    ✖ {i}")



🧠 ИНГРЕДИЕНТЫ ПО ГРУППАМ

🔹 ГРУППА: OTHER
  Известна калорийность (87):
    ✔ ingr_0000000001
    ✔ ingr_0000000002
    ✔ ingr_0000000003
    ✔ ingr_0000000004
    ✔ ingr_0000000005
    ✔ ingr_0000000006
    ✔ ingr_0000000007
    ✔ ingr_0000000008
    ✔ ingr_0000000010
    ✔ ingr_0000000011
    ✔ ingr_0000000012
    ✔ ingr_0000000013
    ✔ ingr_0000000014
    ✔ ingr_0000000015
    ✔ ingr_0000000016
    ✔ ingr_0000000017
    ✔ ingr_0000000021
    ✔ ingr_0000000023
    ✔ ingr_0000000026
    ✔ ingr_0000000027
    ✔ ingr_0000000028
    ✔ ingr_0000000029
    ✔ ingr_0000000030
    ✔ ingr_0000000031
    ✔ ingr_0000000032
    ✔ ingr_0000000033
    ✔ ingr_0000000034
    ✔ ingr_0000000036
    ✔ ingr_0000000037
    ✔ ingr_0000000038
    ✔ ingr_0000000039
    ✔ ingr_0000000042
    ✔ ingr_0000000043
    ✔ ingr_0000000045
    ✔ ingr_0000000046
    ✔ ingr_0000000049
    ✔ ingr_0000000050
    ✔ ingr_0000000054
    ✔ ingr_0000000058
    ✔ ingr_0000000059
    ✔ ingr_0000000073
    ✔ ingr_0000000077
   